In [0]:

dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")

dbutils.widgets.text("source_system", "")
table_name=dbutils.widgets.get("table_name")
landing_timestamp=dbutils.widgets.get("landing_timestamp")

In [0]:
dbutils.widgets.text("source_file_format", "")

source_file_format=dbutils.widgets.get("source_file_format")


In [0]:


source_system=dbutils.widgets.get("source_system")



In [0]:
import os

# Folder where the landing files are
landing_path = f"/Volumes/caresyncdatabricks/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# If the folder (or any parent folder) does not exist, or it is empty, there is nothing to load
if not os.path.isdir(landing_path) or len(os.listdir(landing_path)) == 0:
    print(f"No new files received at: {landing_path}")
    dbutils.notebook.exit("0")

print(f"Landing path found: {landing_path}")

In [0]:
# Clean up the format value: ' CSV ' -> 'csv'
file_format = source_file_format.strip().lower()

# landing_path is defined and validated in the previous cell
print(f"Reading {file_format} files from: {landing_path}")

if file_format == "csv":
    landing_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(landing_path)
else :
    landing_df = spark.read.format("parquet").load(landing_path)


In [0]:
from pyspark.sql.functions import lit
from datetime import datetime
updated_df=landing_df.withColumn("landing_timestamp", lit(landing_timestamp)).withColumn("insert_timestamp", lit(datetime.now()))

In [0]:
updated_df.write.mode("append").saveAsTable(f"caresyncdatabricks.bronze.{table_name}")

In [0]:
%sql
select * from caresyncdatabricks.silver.insurance_providers limit 5